# Debug Forecasting Subtab

This notebook steps through the forecasting tab logic to identify where the `len() of unsized object` error occurs.

In [6]:
import numpy as np
import pickle
from pathlib import Path
import sys

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/bobby/repos/latent-neural-dynamics-modeling


## 1. Configuration - Set your variant and run

In [7]:
# Configure these to match your dashboard selection
VARIANT = "dpad_behavioral_all_blocks_dbs_both"  # Change this to your variant
RUN_TS = "20251201_231446"  # Change this to your run timestamp
SPLIT = "val"  # train, val, or test
TRIAL_IDX = 0  # Which trial to analyze

results_dir = project_root / "results" / VARIANT / RUN_TS
config_path = project_root / "training" / "setups" / f"{VARIANT}.yaml"

print(f"Results dir: {results_dir}")
print(f"Config path: {config_path}")
print(f"Results dir exists: {results_dir.exists()}")
print(f"Config exists: {config_path.exists()}")

Results dir: /home/bobby/repos/latent-neural-dynamics-modeling/results/dpad_behavioral_all_blocks_dbs_both/20251201_231446
Config path: /home/bobby/repos/latent-neural-dynamics-modeling/training/setups/dpad_behavioral_all_blocks_dbs_both.yaml
Results dir exists: True
Config exists: True


## 2. Load Results

In [8]:
# Load the results pickle
pickle_path = results_dir / f"{SPLIT}_results.pkl"
print(f"Pickle path: {pickle_path}")
print(f"Pickle exists: {pickle_path.exists()}")

if pickle_path.exists():
    with open(pickle_path, "rb") as f:
        split_res = pickle.load(f)
    print(f"\nLoaded results with {len(split_res)} keys")
    print(f"Keys: {list(split_res.keys())}")
else:
    print("Pickle not found, trying parquet...")
    # You may need to use the parquet loading logic here

Pickle path: /home/bobby/repos/latent-neural-dynamics-modeling/results/dpad_behavioral_all_blocks_dbs_both/20251201_231446/val_results.pkl
Pickle exists: False
Pickle not found, trying parquet...


## 3. Inspect Data Types

In [9]:
def inspect_value(name, val):
    """Inspect a value's type, shape, and check for len() issues"""
    print(f"\n=== {name} ===")
    print(f"  Type: {type(val)}")

    if val is None:
        print(f"  Value: None")
        return

    if isinstance(val, np.ndarray):
        print(f"  ndim: {val.ndim}")
        print(f"  shape: {val.shape}")
        print(f"  dtype: {val.dtype}")
        if val.ndim == 0:
            print(f"  ⚠️ WARNING: 0-dimensional array! len() will fail!")
            print(f"  Scalar value: {val.item()}")
        else:
            try:
                print(f"  len(): {len(val)}")
            except TypeError as e:
                print(f"  ⚠️ len() ERROR: {e}")
    elif isinstance(val, list):
        print(f"  len(): {len(val)}")
        if len(val) > 0:
            print(f"  First element type: {type(val[0])}")
            if isinstance(val[0], np.ndarray):
                print(
                    f"  First element shape: {val[0].shape if val[0] is not None else 'None'}"
                )
    else:
        try:
            print(f"  len(): {len(val)}")
        except TypeError:
            print(f"  No len() - scalar value: {val}")

In [10]:
# Inspect key forecast-related fields
forecast_keys = [
    "Y_future_pred",
    "Y_future_true",
    "Y_concat_for_plot",
    "Z_future_pred",
    "Z_future_true",
    "Z_concat_for_plot",
    "X_future_pred",
    "pearson_per_channel",
    "pearson_per_channel_Z",
    "metric_m",
    "time_margined",
    "offset",
    "Xp",
]

for key in forecast_keys:
    if key in split_res:
        inspect_value(key, split_res[key])
    else:
        print(f"\n=== {key} ===")
        print(f"  NOT PRESENT in split_res")

NameError: name 'split_res' is not defined

## 4. Step Through render_forecasting_tab Logic

In [ ]:
# Step 1: Check if Y_future_pred exists and has data
print("Step 1: Check Y_future_pred existence")

if "Y_future_pred" in split_res:
    print("  Y_future_pred exists in split_res")
    y_future_pred_data = split_res["Y_future_pred"]
    print(f"  Type: {type(y_future_pred_data)}")

    if y_future_pred_data is not None:
        print("  Y_future_pred is not None")

        # This is where the error likely occurs
        print("\n  Attempting len(split_res['Y_future_pred'])...")
        try:
            data_len = len(y_future_pred_data)
            print(f"  ✓ len() succeeded: {data_len}")
        except TypeError as e:
            print(f"  ✗ len() FAILED: {e}")
            if isinstance(y_future_pred_data, np.ndarray):
                print(f"  ndim: {y_future_pred_data.ndim}")
                print(f"  shape: {y_future_pred_data.shape}")
    else:
        print("  Y_future_pred is None")
else:
    print("  Y_future_pred NOT in split_res")

In [ ]:
# Step 2: Build f_res dictionary
print("Step 2: Build f_res dictionary")

f_res = None

if "Y_future_pred" in split_res and split_res["Y_future_pred"] is not None:
    y_future_pred_data = split_res["Y_future_pred"]

    # Check length safely
    try:
        if isinstance(y_future_pred_data, list):
            data_len = len(y_future_pred_data)
        elif isinstance(y_future_pred_data, np.ndarray):
            if y_future_pred_data.ndim == 0:
                print("  ⚠️ Y_future_pred is 0-d array!")
                data_len = 0
            else:
                data_len = y_future_pred_data.shape[0]
        else:
            data_len = 0

        print(f"  data_len = {data_len}")
        print(f"  TRIAL_IDX = {TRIAL_IDX}")

        if data_len > TRIAL_IDX:
            print(f"  ✓ Can access trial {TRIAL_IDX}")
            f_res = {}

            keys_to_copy = [
                "Y_future_true",
                "Y_future_pred",
                "Y_concat_for_plot",
                "Z_future_true",
                "Z_future_pred",
                "Z_concat_for_plot",
                "X_future_pred",
                "pearson_per_channel",
                "pearson_per_channel_Z",
            ]

            for k in keys_to_copy:
                if k in split_res:
                    print(f"\n  Copying {k}...")
                    val = split_res[k]
                    inspect_value(f"    {k}", val)

                    try:
                        trial_val = val[TRIAL_IDX]
                        print(f"    ✓ Indexed trial {TRIAL_IDX} successfully")
                        if trial_val is not None:
                            f_res[k] = trial_val
                            print(f"    Added to f_res")
                    except (IndexError, TypeError) as e:
                        print(f"    ✗ Error indexing: {e}")
        else:
            print(f"  ✗ Cannot access trial {TRIAL_IDX} (data_len={data_len})")
    except TypeError as e:
        print(f"  ✗ Error checking length: {e}")

print(f"\n\nf_res keys: {list(f_res.keys()) if f_res else 'None'}")

In [ ]:
# Step 3: Get m value
print("Step 3: Get m (forecast horizon) value")

if f_res is not None:
    if "metric_m" in split_res:
        m_val = split_res["metric_m"]
        print(f"  metric_m found: {m_val}")
        print(f"  Type: {type(m_val)}")

        if isinstance(m_val, list) and len(m_val) > 0:
            f_res["m"] = m_val[0]
        else:
            f_res["m"] = m_val
    else:
        print("  metric_m not in split_res, trying config...")
        try:
            from utils.config import get_config

            cfg = get_config(str(config_path))
            m_seconds = cfg.model.forecast.m
            sampling_freq = cfg.data.sampling_frequency
            f_res["m"] = int(m_seconds * sampling_freq)
            print(f"  Got m from config: {f_res['m']}")
        except Exception as e:
            print(f"  ✗ Error getting m from config: {e}")
            f_res["m"] = 0

    print(f"\n  Final m value: {f_res.get('m', 'NOT SET')}")

In [ ]:
# Step 4: Extract and convert arrays
print("Step 4: Extract and convert arrays")

if f_res:
    m = int(f_res.get("m", 0))
    print(f"  m = {m}")

    y_concat = f_res.get("Y_concat_for_plot")
    y_future_true = f_res.get("Y_future_true")
    y_future_pred = f_res.get("Y_future_pred")

    print("\n  Before np.array conversion:")
    inspect_value("    y_concat", y_concat)
    inspect_value("    y_future_true", y_future_true)
    inspect_value("    y_future_pred", y_future_pred)

    if y_concat is not None:
        y_concat = np.array(y_concat)
        print(f"\n  After np.array(y_concat):")
        inspect_value("    y_concat", y_concat)

    if y_future_true is not None:
        y_future_true = np.array(y_future_true)
        print(f"\n  After np.array(y_future_true):")
        inspect_value("    y_future_true", y_future_true)

    if y_future_pred is not None:
        y_future_pred = np.array(y_future_pred)
        print(f"\n  After np.array(y_future_pred):")
        inspect_value("    y_future_pred", y_future_pred)

In [ ]:
# Step 5: Check time arrays
print("Step 5: Check time arrays")

resampled_freq = 60

meta_time_margined = split_res.get("time_margined", [])
offsets = split_res.get("offset", [])

inspect_value("meta_time_margined", meta_time_margined)
inspect_value("offsets", offsets)

if meta_time_margined:
    print(f"\n  Attempting len(meta_time_margined)...")
    try:
        print(f"  ✓ len = {len(meta_time_margined)}")

        if len(meta_time_margined) > TRIAL_IDX:
            t_full = np.array(meta_time_margined[TRIAL_IDX])
            inspect_value(f"  t_full (trial {TRIAL_IDX})", t_full)

            if y_concat is not None:
                print(f"\n  Attempting len(t_full) >= len(y_concat)...")
                try:
                    print(f"  len(t_full) = {len(t_full)}")
                    print(f"  len(y_concat) = {len(y_concat)}")
                except TypeError as e:
                    print(f"  ✗ Error: {e}")
    except TypeError as e:
        print(f"  ✗ Error: {e}")

In [ ]:
# Step 6: Test frequency analysis functions
print("Step 6: Test frequency analysis functions")

from utils.stats import (
    compute_power_spectrum,
    find_dominant_frequencies,
    spectral_correlation,
)

if y_future_true is not None and y_future_pred is not None:
    # Get single channel
    if y_future_true.ndim == 2:
        y_true_ch = y_future_true[:, 0]
    else:
        y_true_ch = y_future_true

    if y_future_pred.ndim == 2:
        y_pred_ch = y_future_pred[:, 0]
    else:
        y_pred_ch = y_future_pred

    inspect_value("y_true_ch", y_true_ch)
    inspect_value("y_pred_ch", y_pred_ch)

    print("\n  Testing compute_power_spectrum...")
    try:
        freqs_true, psd_true = compute_power_spectrum(y_true_ch, 60.0)
        print(f"  ✓ Success")
        inspect_value("    freqs_true", freqs_true)
        inspect_value("    psd_true", psd_true)
    except Exception as e:
        print(f"  ✗ Error: {e}")

    print("\n  Testing find_dominant_frequencies...")
    try:
        dom_freqs, dom_powers = find_dominant_frequencies(
            freqs_true, psd_true, n_peaks=5
        )
        print(f"  ✓ Success")
        inspect_value("    dom_freqs", dom_freqs)
        inspect_value("    dom_powers", dom_powers)
    except Exception as e:
        print(f"  ✗ Error: {e}")
else:
    print("  Cannot test - y_future_true or y_future_pred is None")

In [ ]:
# Step 7: Test the complex functions (band power, coherence)
print("Step 7: Test band power and coherence functions")

from utils.stats import (
    bandpower,
    compare_band_power,
    compute_spectral_coherence,
    CLINICAL_FREQUENCY_BANDS,
)

if y_future_true is not None and y_future_pred is not None:
    if y_future_true.ndim == 2:
        y_true_ch = y_future_true[:, 0]
    else:
        y_true_ch = y_future_true

    if y_future_pred.ndim == 2:
        y_pred_ch = y_future_pred[:, 0]
    else:
        y_pred_ch = y_future_pred

    print("\n  Testing bandpower...")
    try:
        power = bandpower(y_true_ch, 60.0, 3.0, 12.0)
        print(f"  ✓ Success: {power}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback

        traceback.print_exc()

    print("\n  Testing compare_band_power...")
    try:
        band_comparison = compare_band_power(
            y_true_ch, y_pred_ch, 60.0, CLINICAL_FREQUENCY_BANDS
        )
        print(f"  ✓ Success")
        print(f"  Result: {band_comparison}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback

        traceback.print_exc()

    print("\n  Testing compute_spectral_coherence...")
    try:
        freqs_coh, coherence = compute_spectral_coherence(y_true_ch, y_pred_ch, 60.0)
        print(f"  ✓ Success")
        inspect_value("    freqs_coh", freqs_coh)
        inspect_value("    coherence", coherence)
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback

        traceback.print_exc()
else:
    print("  Cannot test - data is None")

In [ ]:
# Step 8: Test Xp handling
print("Step 8: Test Xp (latent states) handling")

Xp = split_res.get("Xp", [])
inspect_value("Xp", Xp)

if Xp:
    print(f"\n  Attempting len(Xp)...")
    try:
        print(f"  ✓ len(Xp) = {len(Xp)}")

        if len(Xp) > TRIAL_IDX and Xp[TRIAL_IDX] is not None:
            x_p_trial = np.array(Xp[TRIAL_IDX])
            inspect_value(f"  x_p_trial (trial {TRIAL_IDX})", x_p_trial)

            if m > 0:
                print(f"\n  Attempting len(x_p_trial) >= m...")
                try:
                    print(f"  len(x_p_trial) = {len(x_p_trial)}")
                    print(f"  m = {m}")
                    print(f"  len(x_p_trial) >= m: {len(x_p_trial) >= m}")
                except TypeError as e:
                    print(f"  ✗ Error: {e}")
    except TypeError as e:
        print(f"  ✗ Error: {e}")

## 5. Plot the Data (if everything works)

In [ ]:
import matplotlib.pyplot as plt

if (
    y_concat is not None
    and y_future_true is not None
    and y_future_pred is not None
    and m > 0
):
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    # Get channel 0
    n_chan = y_concat.shape[1] if y_concat.ndim == 2 else 1
    y_concat_c = y_concat.squeeze() if n_chan == 1 else y_concat[:, 0]
    y_ft_c = y_future_true.squeeze() if n_chan == 1 else y_future_true[:, 0]
    y_fp_c = y_future_pred.squeeze() if n_chan == 1 else y_future_pred[:, 0]

    T = len(y_concat_c)
    Tpast = max(0, T - m)

    # Plot 1: Time series
    ax1 = axes[0]
    ax1.plot(range(Tpast), y_concat_c[:Tpast], label="History", color="blue")
    ax1.plot(range(Tpast, T), y_ft_c, label="True Future", color="green")
    ax1.plot(range(Tpast, T), y_fp_c, label="Forecast", color="red", linestyle="--")
    ax1.axvline(x=Tpast, color="gray", linestyle="--", label="Present")
    ax1.legend()
    ax1.set_title(f"Forecast Plot - Trial {TRIAL_IDX}")
    ax1.set_xlabel("Sample")
    ax1.set_ylabel("Amplitude")

    # Plot 2: Power spectrum
    ax2 = axes[1]
    try:
        freqs_true, psd_true = compute_power_spectrum(y_ft_c, 60.0)
        freqs_pred, psd_pred = compute_power_spectrum(y_fp_c, 60.0)
        ax2.semilogy(freqs_true, psd_true, label="True")
        ax2.semilogy(freqs_pred, psd_pred, label="Predicted", linestyle="--")
        ax2.legend()
        ax2.set_title("Power Spectrum")
        ax2.set_xlabel("Frequency (Hz)")
        ax2.set_ylabel("PSD")
    except Exception as e:
        ax2.text(0.5, 0.5, f"Error: {e}", ha="center", va="center")

    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot - missing required data")
    print(f"  y_concat: {y_concat is not None}")
    print(f"  y_future_true: {y_future_true is not None}")
    print(f"  y_future_pred: {y_future_pred is not None}")
    print(f"  m > 0: {m > 0 if 'm' in dir() else 'undefined'}")

## 6. Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\nVariant: {VARIANT}")
print(f"Run: {RUN_TS}")
print(f"Split: {SPLIT}")
print(f"Trial: {TRIAL_IDX}")

print(f"\nf_res created: {f_res is not None}")
if f_res:
    print(f"f_res keys: {list(f_res.keys())}")
    print(f"m value: {f_res.get('m', 'NOT SET')}")

print(f"\nData shapes:")
if "y_concat" in dir() and y_concat is not None:
    print(f"  y_concat: {y_concat.shape}")
if "y_future_true" in dir() and y_future_true is not None:
    print(f"  y_future_true: {y_future_true.shape}")
if "y_future_pred" in dir() and y_future_pred is not None:
    print(f"  y_future_pred: {y_future_pred.shape}")